In [ ]:
# prepare for Google Colab
import os, sys

_IS_COLAB = "COLAB_RELEASE_TAG" in os.environ and os.path.exists("/content")

if _IS_COLAB:
    import google.colab
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)

    # Enter the project path in Google Drive.
    # This is the path where the project '*.py' file will be imported and where the data, training logs, and models will be saved.
    # e.g. PROJ_PATH = '/content/drive/MyDrive/Colab Notebooks/repository/default_proj'
    PROJ_PATH = "."

    sys.path.append(PROJ_PATH)

    # Install Java, a dependency of pyserini
    !apt install openjdk-21-jre-headless -qq > /dev/null
    os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-21-openjdk-amd64"
    !update-alternatives --set java /usr/lib/jvm/java-21-openjdk-amd64/bin/java
    !java -version

    %pip install faiss-cpu
    %pip install pyserini
    %pip install wget
    %pip install transformers
    %pip install datasets
    %pip install accelerate[torch]
    %pip install evaluate
    %pip install absl-py
    %pip install nltk
    %pip install rouge_score
    %pip install wget
else:
    PROJ_PATH = os.path.abspath(".")
    sys.path.append(PROJ_PATH)
    # Please specify the Java path.
    # If it is different from the path below, refer to the Java installation command above to install Java.
    os.environ.setdefault("JAVA_HOME", os.path.join(sys.prefix, "lib", "jvm"))
    # Local environment is expected to be prepared from README.md.
    # Avoid running pip install automatically on every notebook run.


In [ ]:
import os
import math
import json
import tqdm
import functools
import shutil
import time

import torch
import dataclasses
import transformers
import random
import numpy as np
import evaluate

from model import TransformerConfig, TransformerForCausalLM, TransformerForSequenceClassification

from utils.logger import Logger
from utils.metrics import best_subspan_exact_match
from utils.etc import print_model_statistics

In [ ]:
MODEL_CONTEXT_LENGTH = 1024
ROPE_THETA = 20000.0 # set 50k for 2048 context length, set 20k for 1024 context length
DRIVE_CACHE_PATH = os.path.join(PROJ_PATH, "cache")
LOCAL_CACHE_PATH = "local_cache"
LOG_PATH = os.path.join(PROJ_PATH, "logs")
OUTPUT_PATH = os.path.join(PROJ_PATH, "output")
RESULTS_PATH = os.path.join(PROJ_PATH, "output", "results")
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


os.makedirs(DRIVE_CACHE_PATH, exist_ok=True)
os.makedirs(LOG_PATH, exist_ok=True)
os.makedirs(OUTPUT_PATH, exist_ok=True)
os.makedirs(RESULTS_PATH, exist_ok=True)

# only set matmul precision "high" for ampere and above GPUs(e.g. A100, V100, RTX 30xx, RTX 40xx) unless set "highest"
torch.set_float32_matmul_precision('high')

DO_PRETRAIN = True
DO_FINETUNE_SM = True
DO_FINETUNE_CF = True
DO_FINETUNE_RAG = True
DO_ZEROSHOT_RAG = True
DO_FEWSHOT_RAG = True
DO_CONSTRAINT_RAG = True
DO_PROPOSAL_BASELINES = True
DO_SCRATCH_DOWNSTREAM_BASELINES = True
DO_HARD_NEGATIVE_RAG = True
DO_SUBMISSION = True

rouge = evaluate.load("rouge")

In [ ]:
tokenizer = transformers.AutoTokenizer.from_pretrained("openai-community/gpt2")
tokenizer.deprecation_warnings["Asking-to-pad-a-fast-tokenizer"] = True

additional_special_tokens = {}
if tokenizer.pad_token is None:
    additional_special_tokens["pad_token"] = "<|padding|>"
if tokenizer.eos_token is None:
    additional_special_tokens["eos_token"] = "<|endoftext|>"
if tokenizer.bos_token is None:
    additional_special_tokens["bos_token"] = "<|beginoftext|>"
if additional_special_tokens:
    print(f"Adding special tokens: {additional_special_tokens}")
    tokenizer.add_special_tokens(additional_special_tokens)
tokenizer.padding_side = "left"
tokenizer.model_max_length = MODEL_CONTEXT_LENGTH

In [ ]:
@dataclasses.dataclass
class TrainingConfig(object):
    # Device Setting
    device: str = "cuda"
    model_dtype:str = "cast_bf16"
    # set "cast_bf16" if device is upper than ampere architecture for best performance unless use "amp_fp16" for mixed precision
    # ampere architecture means nvidia a100, a40, a6000 or rtx 3000 series or higher

    # Training setting
    batch_size: int = 32
    eval_batch_size: int = 32
    gradient_accumulation_steps: int = 2
    num_train_epochs: int = 5
    max_steps: int = None

    # Optimizer setting
    optimizer_type: str = "adamw"
    learning_rate: float = 3e-4
    weight_decay: float = 0.01
    warmup_steps: int = 1000
    max_grad_norm:float = 3.0
    lr_scheduler_type: str = "linear"

    # Trainer setting
    metric_for_best_model: str = "loss"
    metric_greater_is_better: bool = False
    # save_interval: int = 1000
    eval_interval: int = 1000
    logging_interval: int = 10
    save_total_limit: int = 5

    logging_path: str = os.path.join(LOG_PATH, "train")
    output_path: str = os.path.join(LOG_PATH, "output")

    def to_dict(self):
        return {
            "model_dtype": self.model_dtype,
            "batch_size": self.batch_size,
            "eval_batch_size": self.eval_batch_size,
            "gradient_accumulation_steps": self.gradient_accumulation_steps,
            "num_train_epochs": self.num_train_epochs,
            "max_steps": self.max_steps,
            "optimizer_type": self.optimizer_type,
            "learning_rate": self.learning_rate,
            "weight_decay": self.weight_decay,
            "warmup_steps": self.warmup_steps,
            "max_grad_norm": self.max_grad_norm,
            "lr_scheduler_type": self.lr_scheduler_type,
            "metric_for_best_model": self.metric_for_best_model,
            "metric_greater_is_better": self.metric_greater_is_better,
            "eval_interval": self.eval_interval,
            "logging_interval": self.logging_interval,
            "save_total_limit": self.save_total_limit,
        }

    def from_dict(self, config_dict):
        for key, value in config_dict.items():
            if hasattr(self, key):
                setattr(self, key, value)
            else:
                raise KeyError(f"Invalid key: {key} in config_dict")
        self.__post_init__()

In [ ]:
@torch.no_grad()
def eval_for_summary(model, dataloader, train_config, tokenizer, max_new_tokens=128):
    if train_config.model_dtype == "cast_bf16":
        model = model.to(torch.bfloat16)

    model = model.to(train_config.device)
    model.eval()
    model.compile()

    prediction = []
    answers = []
    for batch in tqdm.auto.tqdm(dataloader, desc="Evaluating", leave=False):
        batch = {k:v.to(train_config.device) for k,v in batch.items()}
        input_ids = batch.pop("input_ids")
        labels = batch.pop("labels")
        attention_mask = batch.pop("attention_mask")

        generation_input_ids = input_ids.clone()
        generation_attention_mask = attention_mask.clone()
        generation_input_ids[labels.ne(-100)] = tokenizer.pad_token_id
        generation_attention_mask[labels.ne(-100)] = 0

        max_length = max(generation_attention_mask.sum(-1).max().item(), 1)
        refit_generation_input_ids = torch.full((input_ids.shape[0], max_length), tokenizer.pad_token_id, dtype=torch.long, device=input_ids.device)
        refit_generation_attention_mask = torch.zeros((input_ids.shape[0], max_length), dtype=torch.long, device=input_ids.device)
        for i in range(input_ids.shape[0]):
            length = generation_attention_mask[i].sum().item()
            if length == 0:
                continue
            refit_generation_input_ids[i, -length:] = generation_input_ids[i, generation_attention_mask[i].eq(1)]
            refit_generation_attention_mask[i, -length:] = generation_attention_mask[i, generation_attention_mask[i].eq(1)]

        gen_output = model.generate(
            input_ids=refit_generation_input_ids,
            attention_mask=refit_generation_attention_mask,
            max_new_tokens=max_new_tokens,
            return_response_only=True
        )

        labels[labels.eq(-100)] = tokenizer.pad_token_id
        pred = tokenizer.batch_decode(gen_output, skip_special_tokens=True)
        ans = tokenizer.batch_decode(labels, skip_special_tokens=True)

        prediction.extend(pred)
        answers.extend(ans)

    rouge_score = rouge.compute(predictions=prediction, references=answers, use_stemmer=True)
    metrics = {
        "rouge1": rouge_score["rouge1"].item(),
        "rouge2": rouge_score["rouge2"].item(),
        "rougeL": rouge_score["rougeL"].item(),
        "rougeLsum": rouge_score["rougeLsum"].item(),
        "prediction": prediction,
        "answers": answers,
    }

    return metrics

@torch.no_grad()
def eval_for_classification(model, dataloader, train_config):
    if train_config.model_dtype == "cast_bf16":
        model = model.to(torch.bfloat16)

    model = model.to(train_config.device)
    model.eval()
    model.compile()

    prediction = []
    answers = []
    for batch in tqdm.auto.tqdm(dataloader, desc="Evaluating", leave=False):
        batch = {k:v.to(train_config.device) for k,v in batch.items()}
        input_ids = batch.pop("input_ids")
        labels = batch.pop("labels")
        attention_mask = batch.pop("attention_mask")

        logits,_ = model(input_ids=input_ids, attention_mask=attention_mask)
        pred = logits.argmax(-1)

        prediction.extend(pred.tolist())
        answers.extend(labels.tolist())

    prediction = np.array(prediction)
    answers = np.array(answers)

    accuracy = (prediction == answers).sum() / len(answers)

    metrics = {
        "accuracy": accuracy,
        "prediction": prediction,
        "answers": answers,
    }

    return metrics

@torch.no_grad()
def eval_for_rag(model, dataloader, train_config, tokenizer):
    if train_config.model_dtype == "cast_bf16":
        model.model = model.model.to(torch.bfloat16)

    model.model = model.model.to(train_config.device)
    model.model.eval()
    model.model.compile()

    eval_start_time = time.perf_counter()
    uids = []
    questions = []
    predictions = []
    answers = []
    for batch in tqdm.auto.tqdm(dataloader, desc="Evaluating", leave=False):
        uid = batch['uid']
        question = batch['question']
        answer = batch['answers']

        # retrieval_augmented_generate
        extra_kw_args = {
            "pad_token_id": model.tokenizer.eos_token_id,
            "do_sample": False,
        } if isinstance(model.model, transformers.generation.GenerationMixin) else {}
        max_new_tokens = getattr(train_config, "max_new_tokens", 10)
        retrieval_k = getattr(train_config, "retrieval_k", 5)
        outputs = model.retrieval_augmented_generate(
            queries=question,
            qids=uid,
            max_new_tokens=max_new_tokens,
            k=retrieval_k,
            **extra_kw_args
        )
        pred = model.tokenizer.batch_decode(outputs, skip_special_tokens=True, clean_up_tokenization_spaces=False)

        uids.extend(uid)
        questions.extend(question)
        predictions.extend(pred)
        answers.extend(answer)

    # fill here
    #######
    # predictions = [pred.split("\n")[0] for pred in predictions]
    #######


    accuracy = best_subspan_exact_match(predictions, answers)
    rouge_score = rouge.compute(predictions=predictions, references=answers)
    elapsed_seconds = time.perf_counter() - eval_start_time

    metrics = {
        "accuracy": accuracy['acc'],
        "rouge1": rouge_score["rouge1"].item(),
        "rouge2": rouge_score["rouge2"].item(),
        "rougeL": rouge_score["rougeL"].item(),
        "rougeLsum": rouge_score["rougeLsum"].item(),
        "elapsed_seconds": elapsed_seconds,
        "prediction": predictions,
        "answers": answers,
        "uid": uids,
        "question": questions,
    }
    return metrics


In [ ]:
@torch.no_grad()
def eval_loop_for_loss(model, dataloader, train_config, tokenizer):
    model.eval()
    losses = []
    for batch in tqdm.auto.tqdm(dataloader, desc="Evaluating", leave=False):
        batch = {k:v.to(train_config.device) for k,v in batch.items()}

        loss, _ = model(**batch)
        losses.append(loss.item())
    metrics = {
        "loss": sum(losses)/len(losses),
    }
    return metrics


@torch.no_grad()
def eval_loop_for_pretraining(model, dataloader, train_config, tokenizer):
    model.eval()

    losses = []
    ppls = []

    token_correct = 0
    token_total = 0

    for batch in tqdm.auto.tqdm(dataloader, desc="Evaluating", leave=False):
        batch = {k:v.to(train_config.device) for k,v in batch.items()}
        labels = batch.pop("labels")
        logits, _ = model(**batch)

        shifted_logits = logits[..., :-1, :].contiguous()
        shifted_labels = labels[..., 1:].contiguous()

        loss = torch.nn.functional.cross_entropy(shifted_logits.view(-1, shifted_logits.size(-1)), shifted_labels.view(-1),ignore_index=-100, reduction="none")

        loss = loss.view(shifted_labels.size())
        mask = shifted_labels.ne(-100)

        nll_loss = (loss * mask).sum() / mask.sum()
        ppl = torch.exp(nll_loss)

        losses.append(nll_loss.mean())
        ppls.append(ppl.mean())

        token_correct += (shifted_logits.argmax(-1) == shifted_labels).sum().item()
        token_total += shifted_labels.ne(-100).sum().item()

    metrics = {
        "loss": torch.mean(torch.stack(losses)).item(),
        "ppl": torch.mean(torch.stack(ppls)).item(),
        "token_acc": token_correct / token_total,
    }

    return metrics

def train(model, train_dataset, eval_dataset, train_collate_fn, eval_collate_fn, train_config, eval_loop):
    os.makedirs(train_config.logging_path, exist_ok=True)
    os.makedirs(train_config.output_path, exist_ok=True)

    train_model = model if isinstance(model, transformers.PreTrainedModel) else model.model
    if train_config.model_dtype == "cast_bf16":
        train_model = train_model.to(torch.bfloat16)

    print_model_statistics(train_model)

    train_model = train_model.to(train_config.device)
    train_model.train()
    train_model.compile()

    train_loader = torch.utils.data.DataLoader(
        train_dataset,
        batch_size=train_config.batch_size,
        shuffle=True,
        collate_fn=train_collate_fn,
    )
    eval_loader = torch.utils.data.DataLoader(
        eval_dataset,
        batch_size=train_config.eval_batch_size,
        shuffle=False,
        collate_fn=eval_collate_fn,
    )

    num_training_steps = len(train_loader) * train_config.num_train_epochs if train_config.max_steps is None else train_config.max_steps
    num_warmup_steps = train_config.warmup_steps
    num_train_epochs = train_config.num_train_epochs if train_config.max_steps is None else math.ceil(train_config.max_steps / len(train_loader))

    optimizer = torch.optim.AdamW(
        train_model.parameters(),
        lr=train_config.learning_rate,
        weight_decay=train_config.weight_decay,
    )
    scheduler = transformers.get_scheduler(
        train_config.lr_scheduler_type,
        optimizer=optimizer,
        num_warmup_steps=num_warmup_steps,
        num_training_steps=math.ceil(num_training_steps / train_config.gradient_accumulation_steps),
    )

    global_steps = 0

    logger = Logger(log_path=train_config.logging_path)
    with open(os.path.join(train_config.output_path, "train_config.json"), "w") as f:
        json.dump(train_config.to_dict(), f, indent=4)
    logger.log({"train_config": train_config.to_dict()})

    global_pbar = tqdm.auto.tqdm(total=num_training_steps, desc="Training")
    best_models = []
    loss_window = []
    for epoch in range(num_train_epochs):
        train_model.train()
        for batch in train_loader:
            batch = {k:v.to(train_config.device) for k,v in batch.items()}

            with torch.autocast(device_type=train_config.device, dtype=torch.float16, enabled=train_config.model_dtype == "amp_fp16"):
                loss, _ = train_model(**batch)
                l = loss.item()
                loss_window.append(l)
                if len(loss_window) > 100:
                    loss_window.pop(0)

            loss = loss / train_config.gradient_accumulation_steps
            loss.backward()

            should_step = (global_steps + 1) % train_config.gradient_accumulation_steps == 0
            should_step = should_step or (global_steps + 1) == num_training_steps
            if should_step:
                if train_config.max_grad_norm is not None:
                    torch.nn.utils.clip_grad_norm_(train_model.parameters(), train_config.max_grad_norm)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()
            global_steps += 1
            global_pbar.update(1)
            global_pbar.set_postfix({"epoch":epoch, "loss": f"{l:.4f}::{sum(loss_window)/len(loss_window):.4f}", "lr": f"{scheduler.get_last_lr()[0]:.2e}"})

            if global_steps % train_config.logging_interval == 0:
                logger.log({"steps": global_steps, "loss": l,"ave_loss":sum(loss_window)/len(loss_window), "lr": scheduler.get_last_lr()[0]})

            if global_steps % train_config.eval_interval == 0:
                global_pbar.disable = True
                global_pbar.refresh()
                eval_metrics = eval_loop(model, eval_loader, train_config, tokenizer)
                if "prediction" in eval_metrics: eval_metrics.pop("prediction")
                if "answers" in eval_metrics: eval_metrics.pop("answers")
                if "uid" in eval_metrics: eval_metrics.pop("uid")
                if "question" in eval_metrics: eval_metrics.pop("question")

                logger.log({"steps": global_steps, **eval_metrics},is_train=False)
                global_pbar.write(f"====== evaluation {global_steps} ====")
                for k, v in eval_metrics.items():
                    global_pbar.write(f"   {k}: {v}")
                global_pbar.write(f"======================")
                checkpoint = f"step-{global_steps}"

                best_models.append((checkpoint, eval_metrics[train_config.metric_for_best_model]))
                best_models.sort(key=lambda x: x[1], reverse=train_config.metric_greater_is_better)
                if len(best_models) > train_config.save_total_limit:
                    need_to_remove = best_models.pop()
                    if need_to_remove != checkpoint:
                        logger.log({"steps": global_steps, "remove_checkpoint": need_to_remove[0]})
                        shutil.rmtree(os.path.join(train_config.output_path, need_to_remove[0]), ignore_errors=True)
                        logger.log({"steps": global_steps, "save_chechkpoint": checkpoint})
                        os.makedirs(os.path.join(train_config.output_path, checkpoint), exist_ok=True)
                        train_model.save_pretrained(os.path.join(train_config.output_path, checkpoint))
                        tokenizer.save_pretrained(os.path.join(train_config.output_path, checkpoint))
                else:
                    logger.log({"steps": global_steps, "save_chechkpoint": checkpoint})
                    os.makedirs(os.path.join(train_config.output_path, checkpoint), exist_ok=True)
                    train_model.save_pretrained(os.path.join(train_config.output_path, checkpoint))
                    tokenizer.save_pretrained(os.path.join(train_config.output_path, checkpoint))
                global_pbar.disable = False
                global_pbar.refresh()
            if train_config.max_steps is not None and global_steps >= train_config.max_steps:
                break
        if train_config.max_steps is not None and global_steps >= train_config.max_steps:
            break

    eval_metrics = eval_loop(model, eval_loader, train_config, tokenizer)
    logger.log({"steps": global_steps, **eval_metrics},is_train=False)
    global_pbar.write(f"====== Training End ====")

    if "prediction" in eval_metrics: eval_metrics.pop("prediction")
    if "answers" in eval_metrics: eval_metrics.pop("answers")
    if "uid" in eval_metrics: eval_metrics.pop("uid")
    if "question" in eval_metrics: eval_metrics.pop("question")
    for k, v in eval_metrics.items():
        global_pbar.write(f"   {k}: {v}")
    global_pbar.write(f"======================")
    checkpoint = f"step-{global_steps}"

    best_models.append((checkpoint, eval_metrics[train_config.metric_for_best_model]))
    best_models.sort(key=lambda x: x[1], reverse=train_config.metric_greater_is_better)
    if best_models[0][0] != checkpoint:
        shutil.copytree(
            os.path.join(train_config.output_path, best_models[0][0]),
            os.path.join(train_config.output_path, "best_model"),
            dirs_exist_ok=True,
        )
    else:
        train_model.save_pretrained(os.path.join(train_config.output_path, "best_model"))
        tokenizer.save_pretrained(os.path.join(train_config.output_path, "best_model"))


In [ ]:
if DO_PRETRAIN:
    ### About 350M parameters
    # model_config = TransformerConfig(
    #     vocab_size=len(tokenizer),
    #     hidden_size=1024,
    #     intermediate_size=4096,
    #     num_hidden_layers=24,
    #     num_attention_heads=16,
    #     num_key_value_heads=4,
    #     head_dim=64,
    #     max_postion_embeddings=MODEL_CONTEXT_LENGTH,
    #     attention_dropout=0.1,
    #     ffn_dropout=0.05,
    #     pad_token_id=tokenizer.pad_token_id,
    #     bos_token_id=tokenizer.bos_token_id,
    #     eos_token_id=tokenizer.eos_token_id,
    #     rope_theta=ROPE_THETA,
    # )

    ### About 120M parameters
    # model_config = TransformerConfig(
    #     vocab_size=len(tokenizer),
    #     hidden_size=768,
    #     intermediate_size=3072,
    #     num_hidden_layers=12,
    #     num_attention_heads=12,
    #     num_key_value_heads=4,
    #     head_dim=64,
    #     max_postion_embeddings=MODEL_CONTEXT_LENGTH,
    #     attention_dropout=0.1,
    #     ffn_dropout=0.05,
    #     pad_token_id=tokenizer.pad_token_id,
    #     bos_token_id=tokenizer.bos_token_id,
    #     eos_token_id=tokenizer.eos_token_id,
    #     rope_theta=ROPE_THETA,
    # )

    ### About 70M parameters
    model_config = TransformerConfig(
        vocab_size=len(tokenizer),
        hidden_size=512,
        intermediate_size=2048,
        num_hidden_layers=4,
        num_attention_heads=16,
        num_key_value_heads=4,
        head_dim=32,
        max_postion_embeddings=MODEL_CONTEXT_LENGTH,
        attention_dropout=0.1,
        ffn_dropout=0.05,
        pad_token_id=tokenizer.pad_token_id,
        bos_token_id=tokenizer.bos_token_id,
        eos_token_id=tokenizer.eos_token_id,
        rope_theta=ROPE_THETA,
    )

    ### About 30M parameters
    # model_config = TransformerConfig(
    #     vocab_size=len(tokenizer),
    #     hidden_size=384,
    #     intermediate_size=1536,
    #     num_hidden_layers=4,
    #     num_attention_heads=4,
    #     num_key_value_heads=2,
    #     head_dim=32,
    #     max_postion_embeddings=MODEL_CONTEXT_LENGTH,
    #     attention_dropout=0.1,
    #     ffn_dropout=0.05,
    #     pad_token_id=tokenizer.pad_token_id,
    #     bos_token_id=tokenizer.bos_token_id,
    #     eos_token_id=tokenizer.eos_token_id,
    #     rope_theta=ROPE_THETA,
    # )


    from dataset.pretrain import prepare_pretrain_dataset, collate_fn_for_pretrain

    pretraining_config = TrainingConfig(
        device="cuda",
        batch_size=16,
        eval_batch_size=16,
        gradient_accumulation_steps=2,
        num_train_epochs=2,
        max_steps=None,
        optimizer_type="adamw",
        learning_rate=3e-4,
        weight_decay=0.01,
        warmup_steps=1000,
        max_grad_norm=1.0,
        lr_scheduler_type="cosine",
        metric_for_best_model="ppl",
        eval_interval=2500,
        logging_interval=10,
        save_total_limit=3,
        logging_path=os.path.join(LOG_PATH, "pretraining"),
        output_path=os.path.join(OUTPUT_PATH, "pretraining"),
    )

    pretrain_train, pretrain_eval = prepare_pretrain_dataset(tokenizer, max_length=MODEL_CONTEXT_LENGTH, train_sample_size=1048576, eval_sample_size=8096, cache_path=DRIVE_CACHE_PATH)
    model = TransformerForCausalLM(model_config)
    collate_fn = functools.partial(collate_fn_for_pretrain,
                                    tokenizer=tokenizer,
                                    block_length=MODEL_CONTEXT_LENGTH)
    train(model,
        train_dataset=pretrain_train,
        eval_dataset=pretrain_eval,
        train_collate_fn=collate_fn,
        eval_collate_fn=collate_fn,
        train_config=pretraining_config,
        eval_loop=eval_loop_for_pretraining)
    del pretrain_train, pretrain_eval, model
    torch.cuda.empty_cache()


In [ ]:
if DO_FINETUNE_SM:
    from dataset.summary import prepare_summary_dataset, collate_fn_for_summary

    summary_config = TrainingConfig(
        device="cuda",
        batch_size=16,
        eval_batch_size=16,
        gradient_accumulation_steps=2,
        num_train_epochs=3,
        max_steps=None,
        optimizer_type="adamw",
        learning_rate=5e-5,
        weight_decay=0.01,
        warmup_steps=1000,
        max_grad_norm=1.0,
        lr_scheduler_type="cosine",
        metric_for_best_model="loss",
        eval_interval=2500,
        logging_interval=10,
        save_total_limit=3,
        logging_path=os.path.join(LOG_PATH, "summary"),
        output_path=os.path.join(OUTPUT_PATH, "summary"),
    )

    model = TransformerForCausalLM.from_pretrained(os.path.join(OUTPUT_PATH, "pretraining", "best_model"))
    summary_train, summary_eval = prepare_summary_dataset(tokenizer)
    collate_fn = functools.partial(collate_fn_for_summary,
                                    tokenizer=tokenizer)
    train(model,
        train_dataset=summary_train,
        eval_dataset=summary_eval,
        train_collate_fn= collate_fn,
        eval_collate_fn=collate_fn,
        train_config=summary_config,
        eval_loop=eval_loop_for_loss)

    del model
    torch.cuda.empty_cache()
    model = TransformerForCausalLM.from_pretrained(os.path.join(OUTPUT_PATH, "summary", "best_model"))
    eval_loader = torch.utils.data.DataLoader(
        summary_eval,
        batch_size=summary_config.eval_batch_size,
        shuffle=False,
        collate_fn=collate_fn,
    )
    metrics = eval_for_summary(model, eval_loader, summary_config, tokenizer, max_new_tokens=128)
    prediction = metrics.pop("prediction")
    answers = metrics.pop("answers")
    with open(os.path.join(RESULTS_PATH,"summary_score.json"), "w") as f:
        json.dump(metrics, f, indent=4)
    with open(os.path.join(RESULTS_PATH,"summary_output.txt"), "w") as f:
        for p,a in zip(prediction, answers):
            a = str(a).replace("\r", " ").replace("\n", " ").strip()
            p = str(p).replace("\r", " ").replace("\n", " ").strip()
            f.write(f"{a}\t{p}\n")

    del summary_train, summary_eval, model
    torch.cuda.empty_cache()

In [ ]:
if DO_FINETUNE_CF:
    from dataset.classification import prepare_classification_dataset, collate_fn_for_classification

    classification_config = TrainingConfig(
        device="cuda",
        batch_size=32,
        eval_batch_size=32,
        gradient_accumulation_steps=1,
        num_train_epochs=5,
        max_steps=None,
        optimizer_type="adamw",
        learning_rate=5e-5,
        weight_decay=0.01,
        warmup_steps=1000,
        max_grad_norm=1.0,
        lr_scheduler_type="cosine",
        metric_for_best_model="loss",
        eval_interval=2500,
        logging_interval=10,
        save_total_limit=3,
        logging_path=os.path.join(LOG_PATH, "classification"),
        output_path=os.path.join(OUTPUT_PATH, "classification"),
    )

    model = TransformerForSequenceClassification.from_pretrained(os.path.join(OUTPUT_PATH, "pretraining", "best_model"),num_labels=20)
    class_train, class_eval = prepare_classification_dataset(tokenizer)
    collate_fn = functools.partial(collate_fn_for_classification,
                                    tokenizer=tokenizer)
    train(model,
        train_dataset=class_train,
        eval_dataset=class_eval,
        train_collate_fn=collate_fn,
        eval_collate_fn=collate_fn,
        train_config=classification_config,
        eval_loop=eval_loop_for_loss)

    del model
    torch.cuda.empty_cache()
    model = TransformerForSequenceClassification.from_pretrained(os.path.join(OUTPUT_PATH, "classification", "best_model"),num_labels=20)
    eval_loader = torch.utils.data.DataLoader(
        class_eval,
        batch_size=classification_config.eval_batch_size,
        shuffle=False,
        collate_fn=collate_fn,
    )
    metrics = eval_for_classification(model, eval_loader, classification_config)
    prediction = metrics.pop("prediction")
    answers = metrics.pop("answers")
    with open(os.path.join(RESULTS_PATH,"classification_score.json"), "w") as f:
        json.dump(metrics, f, indent=4)
    with open(os.path.join(RESULTS_PATH,"classification_output.txt"), "w") as f:
        for p,a in zip(prediction, answers):
            f.write(f"{a}\t{p}\n")

    del class_train, class_eval, model
    torch.cuda.empty_cache()

In [ ]:
if DO_FINETUNE_RAG:
    from dataset.rag import donwload_dataset_nq_open_dpr, RAGDataset, RAGCollator
    from pyserini.search.lucene import LuceneSearcher
    from model_rag import ModelRAG

    rag_config = TrainingConfig(
        device="cuda",
        batch_size=16,
        eval_batch_size=16,
        gradient_accumulation_steps=1,
        num_train_epochs=3,
        max_steps=None,
        optimizer_type="adamw",
        learning_rate=5e-5,
        weight_decay=0.01,
        warmup_steps=1000,
        max_grad_norm=1.0,
        lr_scheduler_type="cosine",
        metric_for_best_model="accuracy",
        metric_greater_is_better=True,
        eval_interval=2500,
        logging_interval=10,
        save_total_limit=3,
        logging_path=os.path.join(LOG_PATH, "rag"),
        output_path=os.path.join(OUTPUT_PATH, "rag"),
    )

    rag_cache_path = os.path.join(LOCAL_CACHE_PATH, "rag")
    donwload_dataset_nq_open_dpr(rag_cache_path)

    PREBUILT_INDEX_NAME_BM25 = "wikipedia-dpr-100w"
    LOCAL_INDEX_NAME_BM25 = os.path.join(rag_cache_path, "lucene-index.wikipedia-dpr-100w.20210120.d1b9e6")
    # it might take a 10-20 minutes to download the index, recommand to use drive cache, if drive capacity is enough

    try:
        searcher_bm25 = LuceneSearcher(index_dir=LOCAL_INDEX_NAME_BM25)
    except:
        import shutil
        searcher_bm25 = LuceneSearcher.from_prebuilt_index(prebuilt_index_name=PREBUILT_INDEX_NAME_BM25)
        index_dir = searcher_bm25.index_dir
        shutil.move(index_dir, LOCAL_INDEX_NAME_BM25)
        searcher_bm25 = LuceneSearcher(index_dir=LOCAL_INDEX_NAME_BM25)

    model = TransformerForCausalLM.from_pretrained(os.path.join(OUTPUT_PATH, "pretraining", "best_model"))
    train_dataset = RAGDataset(
        tokenizer=tokenizer,
        is_train=True,
        dataset_path=os.path.join(rag_cache_path,"data","nq_open_dpr","nq_train"),
        num_samples=None
    )
    eval_dataset = RAGDataset(
        tokenizer=tokenizer,
        is_train=False,
        dataset_path=os.path.join(rag_cache_path,"data","nq_open_dpr","nq_dev"),
    )
    train_collator = RAGCollator(
        tokenizer=tokenizer,
        is_train=True,
    )
    eval_collator = RAGCollator(
        tokenizer=tokenizer,
        is_train=False,
    )

    model_rag = ModelRAG()
    model_rag.set_model(model)
    model_rag.set_tokenizer(tokenizer)
    model_rag.set_retriever(searcher_bm25)


    train(
        model_rag,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        train_collate_fn=train_collator,
        eval_collate_fn=eval_collator,
        train_config=rag_config,
        eval_loop=eval_for_rag,
    )

    model_rag.set_model(None)
    del model
    torch.cuda.empty_cache()
    model = TransformerForCausalLM.from_pretrained(os.path.join(OUTPUT_PATH, "rag", "best_model"))

    model_rag.set_model(model)

    eval_loader = torch.utils.data.DataLoader(
        eval_dataset,
        batch_size=rag_config.eval_batch_size,
        shuffle=False,
        collate_fn=eval_collator,
    )
    metrics = eval_for_rag(model_rag, eval_loader, rag_config, tokenizer)
    prediction = metrics.pop("prediction")
    answers = metrics.pop("answers")
    uid = metrics.pop("uid")
    questions = metrics.pop("question")
    
    print(f"====== evaluation ====")
    for k, v in metrics.items():
        print(f"   {k}: {v}")
    print(f"======================")
          
    with open(os.path.join(RESULTS_PATH,"rag_score.json"), "w") as f:
        json.dump(metrics, f, indent=4)
    with open(os.path.join(RESULTS_PATH,"rag_output.txt"), "w") as f:
        for u,q,p,a in zip(uid, questions, prediction, answers):
            q = str(q).replace("\r", " ").replace("\n", " ").strip()
            a = str(a).replace("\r", " ").replace("\n", " ").strip()
            p = str(p).replace("\r", " ").replace("\n", " ").strip()
            f.write(f"{u}\t{q}\t{a}\t{p}\n")

    del train_dataset, eval_dataset, model, model_rag
    torch.cuda.empty_cache()

In [ ]:
if DO_SCRATCH_DOWNSTREAM_BASELINES:
    from dataset.summary import prepare_summary_dataset, collate_fn_for_summary
    from dataset.classification import prepare_classification_dataset, collate_fn_for_classification
    from dataset.rag import donwload_dataset_nq_open_dpr, RAGDataset, RAGCollator
    from pyserini.search.lucene import LuceneSearcher
    from model_rag import ModelRAG

    def build_scratch_model_config():
        return TransformerConfig(
            vocab_size=len(tokenizer),
            hidden_size=512,
            intermediate_size=2048,
            num_hidden_layers=4,
            num_attention_heads=16,
            num_key_value_heads=4,
            head_dim=32,
            max_position_embeddings=MODEL_CONTEXT_LENGTH,
            attention_dropout=0.1,
            ffn_dropout=0.05,
            pad_token_id=tokenizer.pad_token_id,
            bos_token_id=tokenizer.bos_token_id,
            eos_token_id=tokenizer.eos_token_id,
            rope_theta=ROPE_THETA,
        )

    scratch_model_config = build_scratch_model_config()

    # Summary without language-model pretraining.
    summary_config_scratch = TrainingConfig(
        device="cuda",
        batch_size=16,
        eval_batch_size=16,
        gradient_accumulation_steps=2,
        num_train_epochs=3,
        max_steps=None,
        optimizer_type="adamw",
        learning_rate=5e-5,
        weight_decay=0.01,
        warmup_steps=1000,
        max_grad_norm=1.0,
        lr_scheduler_type="cosine",
        metric_for_best_model="loss",
        eval_interval=2500,
        logging_interval=10,
        save_total_limit=3,
        logging_path=os.path.join(LOG_PATH, "summary_scratch"),
        output_path=os.path.join(OUTPUT_PATH, "summary_scratch"),
    )
    summary_train, summary_eval = prepare_summary_dataset(
        tokenizer,
        cache_path=os.path.join(DRIVE_CACHE_PATH, "summary_scratch_full"),
    )
    summary_collate_fn = functools.partial(collate_fn_for_summary, tokenizer=tokenizer)
    model = TransformerForCausalLM(scratch_model_config)
    train(
        model,
        train_dataset=summary_train,
        eval_dataset=summary_eval,
        train_collate_fn=summary_collate_fn,
        eval_collate_fn=summary_collate_fn,
        train_config=summary_config_scratch,
        eval_loop=eval_loop_for_loss,
    )
    del model
    torch.cuda.empty_cache()
    model = TransformerForCausalLM.from_pretrained(os.path.join(OUTPUT_PATH, "summary_scratch", "best_model"))
    eval_loader = torch.utils.data.DataLoader(
        summary_eval,
        batch_size=summary_config_scratch.eval_batch_size,
        shuffle=False,
        collate_fn=summary_collate_fn,
    )
    metrics = eval_for_summary(model, eval_loader, summary_config_scratch, tokenizer, max_new_tokens=128)
    prediction = metrics.pop("prediction")
    answers = metrics.pop("answers")
    metrics.update({
        "variant": "summary_scratch_no_pretraining",
        "variant_description": "Summary finetuning from random initialization without LM pretraining",
        "train_samples": "full",
        "eval_samples": len(summary_eval),
        "max_steps": "full_epochs",
    })
    with open(os.path.join(RESULTS_PATH, "summary_scratch_score.json"), "w") as f:
        json.dump(metrics, f, indent=4)
    with open(os.path.join(RESULTS_PATH, "summary_scratch_output.txt"), "w") as f:
        for p, a in zip(prediction, answers):
            f.write(f"{str(a).replace(chr(10), ' ').replace(chr(13), ' ').strip()}\t{str(p).replace(chr(10), ' ').replace(chr(13), ' ').strip()}\n")
    del summary_train, summary_eval, model
    torch.cuda.empty_cache()

    # Classification without language-model pretraining.
    classification_config_scratch = TrainingConfig(
        device="cuda",
        batch_size=32,
        eval_batch_size=32,
        gradient_accumulation_steps=1,
        num_train_epochs=5,
        max_steps=None,
        optimizer_type="adamw",
        learning_rate=5e-5,
        weight_decay=0.01,
        warmup_steps=1000,
        max_grad_norm=1.0,
        lr_scheduler_type="cosine",
        metric_for_best_model="loss",
        eval_interval=2500,
        logging_interval=10,
        save_total_limit=3,
        logging_path=os.path.join(LOG_PATH, "classification_scratch"),
        output_path=os.path.join(OUTPUT_PATH, "classification_scratch"),
    )
    class_train, class_eval = prepare_classification_dataset(
        tokenizer,
        cache_path=os.path.join(DRIVE_CACHE_PATH, "classification_scratch_full"),
    )
    class_collate_fn = functools.partial(collate_fn_for_classification, tokenizer=tokenizer)
    scratch_classification_config = build_scratch_model_config()
    scratch_classification_config.num_labels = 20
    model = TransformerForSequenceClassification(scratch_classification_config)
    train(
        model,
        train_dataset=class_train,
        eval_dataset=class_eval,
        train_collate_fn=class_collate_fn,
        eval_collate_fn=class_collate_fn,
        train_config=classification_config_scratch,
        eval_loop=eval_loop_for_loss,
    )
    del model
    torch.cuda.empty_cache()
    model = TransformerForSequenceClassification.from_pretrained(os.path.join(OUTPUT_PATH, "classification_scratch", "best_model"), num_labels=20)
    eval_loader = torch.utils.data.DataLoader(
        class_eval,
        batch_size=classification_config_scratch.eval_batch_size,
        shuffle=False,
        collate_fn=class_collate_fn,
    )
    metrics = eval_for_classification(model, eval_loader, classification_config_scratch)
    prediction = metrics.pop("prediction")
    answers = metrics.pop("answers")
    metrics.update({
        "variant": "classification_scratch_no_pretraining",
        "variant_description": "Classification finetuning from random initialization without LM pretraining",
        "train_samples": "full",
        "eval_samples": len(class_eval),
        "max_steps": "full_epochs",
    })
    with open(os.path.join(RESULTS_PATH, "classification_scratch_score.json"), "w") as f:
        json.dump(metrics, f, indent=4)
    with open(os.path.join(RESULTS_PATH, "classification_scratch_output.txt"), "w") as f:
        for p, a in zip(prediction, answers):
            f.write(f"{a}\t{p}\n")
    del class_train, class_eval, model
    torch.cuda.empty_cache()

    # RAG without language-model pretraining.
    rag_config_scratch = TrainingConfig(
        device="cuda",
        batch_size=16,
        eval_batch_size=16,
        gradient_accumulation_steps=1,
        num_train_epochs=3,
        max_steps=None,
        optimizer_type="adamw",
        learning_rate=5e-5,
        weight_decay=0.01,
        warmup_steps=1000,
        max_grad_norm=1.0,
        lr_scheduler_type="cosine",
        metric_for_best_model="accuracy",
        metric_greater_is_better=True,
        eval_interval=2500,
        logging_interval=10,
        save_total_limit=3,
        logging_path=os.path.join(LOG_PATH, "rag_scratch"),
        output_path=os.path.join(OUTPUT_PATH, "rag_scratch"),
    )
    rag_cache_path = os.path.join(LOCAL_CACHE_PATH, "rag")
    donwload_dataset_nq_open_dpr(rag_cache_path)
    PREBUILT_INDEX_NAME_BM25 = "wikipedia-dpr-100w"
    LOCAL_INDEX_NAME_BM25 = os.path.join(rag_cache_path, "lucene-index.wikipedia-dpr-100w.20210120.d1b9e6")
    if "searcher_bm25" not in globals():
        try:
            searcher_bm25 = LuceneSearcher(index_dir=LOCAL_INDEX_NAME_BM25)
        except:
            import shutil
            searcher_bm25 = LuceneSearcher.from_prebuilt_index(prebuilt_index_name=PREBUILT_INDEX_NAME_BM25)
            index_dir = searcher_bm25.index_dir
            shutil.move(index_dir, LOCAL_INDEX_NAME_BM25)
            searcher_bm25 = LuceneSearcher(index_dir=LOCAL_INDEX_NAME_BM25)
    train_dataset = RAGDataset(
        tokenizer=tokenizer,
        is_train=True,
        dataset_path=os.path.join(rag_cache_path, "data", "nq_open_dpr", "nq_train"),
    )
    eval_dataset = RAGDataset(
        tokenizer=tokenizer,
        is_train=False,
        dataset_path=os.path.join(rag_cache_path, "data", "nq_open_dpr", "nq_dev"),
    )
    train_collator = RAGCollator(tokenizer=tokenizer, is_train=True)
    eval_collator = RAGCollator(tokenizer=tokenizer, is_train=False)
    model = TransformerForCausalLM(scratch_model_config)
    model_rag = ModelRAG()
    model_rag.set_model(model)
    model_rag.set_tokenizer(tokenizer)
    model_rag.set_retriever(searcher_bm25)
    train(
        model_rag,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        train_collate_fn=train_collator,
        eval_collate_fn=eval_collator,
        train_config=rag_config_scratch,
        eval_loop=eval_for_rag,
    )
    model_rag.set_model(None)
    del model
    torch.cuda.empty_cache()
    model = TransformerForCausalLM.from_pretrained(os.path.join(OUTPUT_PATH, "rag_scratch", "best_model"))
    model_rag.set_model(model)
    eval_loader = torch.utils.data.DataLoader(
        eval_dataset,
        batch_size=rag_config_scratch.eval_batch_size,
        shuffle=False,
        collate_fn=eval_collator,
    )
    metrics = eval_for_rag(model_rag, eval_loader, rag_config_scratch, tokenizer)
    prediction = metrics.pop("prediction")
    answers = metrics.pop("answers")
    uid = metrics.pop("uid")
    questions = metrics.pop("question")
    metrics.update({
        "variant": "rag_scratch_no_pretraining",
        "variant_description": "RAG finetuning from random initialization without LM pretraining",
        "train_samples": "full",
        "num_samples": len(eval_dataset),
        "max_steps": "full_epochs",
    })
    with open(os.path.join(RESULTS_PATH, "rag_scratch_score.json"), "w") as f:
        json.dump(metrics, f, indent=4)
    with open(os.path.join(RESULTS_PATH, "rag_scratch_output.txt"), "w") as f:
        for u, q, p, a in zip(uid, questions, prediction, answers):
            answer = " | ".join(a) if isinstance(a, list) else str(a)
            f.write(f"{u}\t{str(q).replace(chr(10), ' ').replace(chr(13), ' ').strip()}\t{answer.replace(chr(10), ' ').replace(chr(13), ' ').strip()}\t{str(p).replace(chr(10), ' ').replace(chr(13), ' ').strip()}\n")
    del train_dataset, eval_dataset, model, model_rag
    torch.cuda.empty_cache()


In [ ]:
# For colab
# from huggingface_hub import notebook_login, login
# notebook_login()

In [ ]:
if DO_ZEROSHOT_RAG:
    from huggingface_hub import get_token
    if get_token() is None:
        raise RuntimeError("Hugging Face token not found. Run `hf auth login` before Llama zero-shot RAG.")
    from dataset.rag import donwload_dataset_nq_open_dpr, RAGDataset, RAGCollator
    from pyserini.search.lucene import LuceneSearcher
    from model_rag import ModelRAG

    rag_cache_path = os.path.join(LOCAL_CACHE_PATH, "rag")
    donwload_dataset_nq_open_dpr(rag_cache_path)

    PREBUILT_INDEX_NAME_BM25 = "wikipedia-dpr-100w"
    LOCAL_INDEX_NAME_BM25 = os.path.join(rag_cache_path, "lucene-index.wikipedia-dpr-100w.20210120.d1b9e6")
    # it might take a 10-20 minutes to download the index, recommand to use drive cache, if drive capacity is enough

    try:
        searcher_bm25 = LuceneSearcher(index_dir=LOCAL_INDEX_NAME_BM25)
    except:
        import shutil
        searcher_bm25 = LuceneSearcher.from_prebuilt_index(prebuilt_index_name=PREBUILT_INDEX_NAME_BM25)
        index_dir = searcher_bm25.index_dir
        shutil.move(index_dir, LOCAL_INDEX_NAME_BM25)
        searcher_bm25 = LuceneSearcher(index_dir=LOCAL_INDEX_NAME_BM25)


    model_name_or_path = "meta-llama/Llama-3.2-1B-Instruct"
    llm_model = transformers.AutoModelForCausalLM.from_pretrained(model_name_or_path,
                                                                  torch_dtype=torch.bfloat16,
                                                                  device_map={"": 0},
                                                                  low_cpu_mem_usage=True,)
    llm_tokenizer = transformers.AutoTokenizer.from_pretrained(model_name_or_path)
    llm_batch_size = 4
    llm_tokenizer.pad_token = llm_tokenizer.eos_token
    llm_tokenizer.padding_side = "left"


    eval_dataset = RAGDataset(
        tokenizer=llm_tokenizer,
        is_train=False,
        dataset_path=os.path.join(rag_cache_path,"data","nq_open_dpr","nq_dev"),
    )
    eval_collator = RAGCollator(
        tokenizer=llm_tokenizer,
        is_train=False,
    )

    model_rag = ModelRAG()
    model_rag.set_model(llm_model)
    model_rag.set_tokenizer(llm_tokenizer)
    model_rag.set_retriever(searcher_bm25)

    eval_loader = torch.utils.data.DataLoader(
        eval_dataset,
        batch_size=llm_batch_size,
        shuffle=False,
        collate_fn=eval_collator,
    )
    class rag_config:
        device = "cuda"
        model_dtype = "cast_bf16"
        max_new_tokens = 32
        retrieval_k = 5

    metrics = eval_for_rag(model_rag, eval_loader, rag_config, tokenizer)
    prediction = metrics.pop("prediction")
    answers = metrics.pop("answers")
    uid = metrics.pop("uid")
    questions = metrics.pop("question")
    
    metrics.update({
        "num_samples": len(eval_dataset),
        "retrieval_top_k": rag_config.retrieval_k,
        "max_new_tokens": rag_config.max_new_tokens,
        "variant": "llm_rag",
        "variant_description": "Llama zero-shot RAG with BM25 top-5 retrieval and max_new_tokens=32",
    })

    print(f"====== evaluation ====")
    for k, v in metrics.items():
        print(f"   {k}: {v}")
    print(f"======================")
        
    with open(os.path.join(RESULTS_PATH,"llm_rag_score.json"), "w") as f:
        json.dump(metrics, f, indent=4)
    with open(os.path.join(RESULTS_PATH,"llm_rag_output.txt"), "w") as f:
        for u,q,p,a in zip(uid, questions, prediction, answers):
            q = str(q).replace("\r", " ").replace("\n", " ").strip()
            a = str(a).replace("\r", " ").replace("\n", " ").strip()
            p = str(p).replace("\r", " ").replace("\n", " ").strip()
            f.write(f"{u}\t{q}\t{a}\t{p}\n")


In [ ]:
if DO_FEWSHOT_RAG:
    from huggingface_hub import get_token
    if get_token() is None:
        raise RuntimeError("Hugging Face token not found. Run `hf auth login` before Llama few-shot RAG.")
    from dataset.rag import donwload_dataset_nq_open_dpr, RAGDataset, RAGCollator
    from pyserini.search.lucene import LuceneSearcher
    from model_rag import ModelRAG

    class FewShotModelRAG(ModelRAG):
        def make_augmented_inputs_for_generate(self, queries, qids, k=5):
            list_passages, _ = self.search(queries, qids, k=k)
            few_shot_prefix = (
                "Answer each question using only the provided passages. Write only the short answer.\n\n"
                "Example 1:\n"
                "Title: Nineteen Eighty-Four\n"
                "Passage: Nineteen Eighty-Four is a dystopian novel by English writer George Orwell.\n"
                "Question: who wrote the novel 1984\n"
                "Answer: George Orwell\n\n"
                "Example 2:\n"
                "Title: Berlin Wall\n"
                "Passage: The Berlin Wall was opened on 9 November 1989.\n"
                "Question: when did the berlin wall fall\n"
                "Answer: November 9, 1989\n\n"
                "Now answer the question using the passages below.\n"
            )
            list_input_text_without_answer = []
            for query, passages in zip(queries, list_passages):
                context_blocks = []
                for passage in passages:
                    if isinstance(passage, dict):
                        title = passage.get("title") or ""
                        text = (
                            passage.get("text")
                            or passage.get("contents")
                            or passage.get("passage")
                            or ""
                        )
                    else:
                        title = ""
                        text = str(passage)
                    context_blocks.append(f"Title: {title}\nPassage: {text}")
                context = "\n".join(context_blocks)
                input_text = f"{few_shot_prefix}{context}\nQuestion: {query}\nAnswer:"
                list_input_text_without_answer.append(input_text)
            return list_input_text_without_answer

    rag_cache_path = os.path.join(LOCAL_CACHE_PATH, "rag")
    donwload_dataset_nq_open_dpr(rag_cache_path)

    PREBUILT_INDEX_NAME_BM25 = "wikipedia-dpr-100w"
    LOCAL_INDEX_NAME_BM25 = os.path.join(rag_cache_path, "lucene-index.wikipedia-dpr-100w.20210120.d1b9e6")

    try:
        searcher_bm25 = LuceneSearcher(index_dir=LOCAL_INDEX_NAME_BM25)
    except:
        import shutil
        searcher_bm25 = LuceneSearcher.from_prebuilt_index(prebuilt_index_name=PREBUILT_INDEX_NAME_BM25)
        index_dir = searcher_bm25.index_dir
        shutil.move(index_dir, LOCAL_INDEX_NAME_BM25)
        searcher_bm25 = LuceneSearcher(index_dir=LOCAL_INDEX_NAME_BM25)

    model_name_or_path = "meta-llama/Llama-3.2-1B-Instruct"
    llm_model = transformers.AutoModelForCausalLM.from_pretrained(
        model_name_or_path,
        torch_dtype=torch.bfloat16,
        device_map={"": 0},
        low_cpu_mem_usage=True,
    )
    llm_tokenizer = transformers.AutoTokenizer.from_pretrained(model_name_or_path)
    llm_batch_size = 4
    llm_tokenizer.pad_token = llm_tokenizer.eos_token
    llm_tokenizer.padding_side = "left"

    eval_dataset = RAGDataset(
        tokenizer=llm_tokenizer,
        is_train=False,
        dataset_path=os.path.join(rag_cache_path, "data", "nq_open_dpr", "nq_dev"),
    )
    eval_collator = RAGCollator(
        tokenizer=llm_tokenizer,
        is_train=False,
    )

    model_rag = FewShotModelRAG()
    model_rag.set_model(llm_model)
    model_rag.set_tokenizer(llm_tokenizer)
    model_rag.set_retriever(searcher_bm25)

    eval_loader = torch.utils.data.DataLoader(
        eval_dataset,
        batch_size=llm_batch_size,
        shuffle=False,
        collate_fn=eval_collator,
    )

    class fewshot_rag_config:
        device = "cuda"
        model_dtype = "cast_bf16"
        max_new_tokens = 32
        retrieval_k = 5

    metrics = eval_for_rag(model_rag, eval_loader, fewshot_rag_config, tokenizer)
    prediction = metrics.pop("prediction")
    answers = metrics.pop("answers")
    uid = metrics.pop("uid")
    questions = metrics.pop("question")

    metrics.update({
        "num_samples": len(eval_dataset),
        "retrieval_top_k": fewshot_rag_config.retrieval_k,
        "max_new_tokens": fewshot_rag_config.max_new_tokens,
        "num_few_shot_examples": 2,
        "variant": "llm_fewshot_rag",
        "variant_description": "Llama few-shot RAG baseline with BM25 top-5 retrieval, two in-prompt QA examples, and max_new_tokens=32",
    })

    print("====== few-shot Llama RAG evaluation ====")
    for k, v in metrics.items():
        print(f"   {k}: {v}")
    print("=========================================")

    with open(os.path.join(RESULTS_PATH, "llm_fewshot_rag_score.json"), "w") as f:
        json.dump(metrics, f, indent=4)
    with open(os.path.join(RESULTS_PATH, "llm_fewshot_rag_output.txt"), "w") as f:
        for u, q, p, a in zip(uid, questions, prediction, answers):
            q = str(q).replace("\r", " ").replace("\n", " ").strip()
            a = str(a).replace("\r", " ").replace("\n", " ").strip()
            p = str(p).replace("\r", " ").replace("\n", " ").strip()
            f.write(f"{u}\t{q}\t{a}\t{p}\n")

    del model_rag, llm_model, llm_tokenizer, eval_dataset, eval_loader
    torch.cuda.empty_cache()


In [ ]:
if DO_CONSTRAINT_RAG:
    from huggingface_hub import get_token
    if get_token() is None:
        raise RuntimeError("Hugging Face token not found. Run `hf auth login` before ConstraintRAG evaluation.")
    from dataset.rag import donwload_dataset_nq_open_dpr, RAGDataset, RAGCollator
    from pyserini.search.lucene import LuceneSearcher
    import importlib
    import constraint_rag
    constraint_rag = importlib.reload(constraint_rag)
    ConstraintRAGConfig = constraint_rag.ConstraintRAGConfig
    ConstraintRAG = constraint_rag.ConstraintRAG
    extract_answer_from_json = constraint_rag.extract_answer_from_json
    extract_first_answer_candidate = constraint_rag.extract_first_answer_candidate
    get_evidence_candidates_text = constraint_rag.get_evidence_candidates_text
    strip_evidence_candidates = constraint_rag.strip_evidence_candidates
    from extra_experiments import compute_rag_metrics, write_constraint_output, write_rag_output

    rag_cache_path = os.path.join(LOCAL_CACHE_PATH, "rag")
    os.makedirs(rag_cache_path, exist_ok=True)
    donwload_dataset_nq_open_dpr(rag_cache_path)

    PREBUILT_INDEX_NAME_BM25 = "wikipedia-dpr-100w"
    LOCAL_INDEX_NAME_BM25 = os.path.join(rag_cache_path, "lucene-index.wikipedia-dpr-100w.20210120.d1b9e6")

    if "searcher_bm25" not in globals():
        try:
            searcher_bm25 = LuceneSearcher(index_dir=LOCAL_INDEX_NAME_BM25)
        except:
            import shutil
            searcher_bm25 = LuceneSearcher.from_prebuilt_index(prebuilt_index_name=PREBUILT_INDEX_NAME_BM25)
            index_dir = searcher_bm25.index_dir
            shutil.move(index_dir, LOCAL_INDEX_NAME_BM25)
            searcher_bm25 = LuceneSearcher(index_dir=LOCAL_INDEX_NAME_BM25)

    if "llm_model" not in globals() or "llm_tokenizer" not in globals():
        model_name_or_path = "meta-llama/Llama-3.2-1B-Instruct"
        llm_model = transformers.AutoModelForCausalLM.from_pretrained(
            model_name_or_path,
            torch_dtype=torch.bfloat16,
            device_map={"": 0},
            low_cpu_mem_usage=True,
        )
        llm_tokenizer = transformers.AutoTokenizer.from_pretrained(model_name_or_path)
        llm_tokenizer.pad_token = llm_tokenizer.eos_token
        llm_tokenizer.padding_side = "left"

    eval_dataset = RAGDataset(
        tokenizer=llm_tokenizer,
        is_train=False,
        dataset_path=os.path.join(rag_cache_path, "data", "nq_open_dpr", "nq_dev"),
        num_samples=None,
    )
    eval_collator = RAGCollator(tokenizer=llm_tokenizer, is_train=False)
    eval_loader = torch.utils.data.DataLoader(
        eval_dataset,
        batch_size=4,
        shuffle=False,
        collate_fn=eval_collator,
    )
    diagnostic_cache_path = os.path.join(rag_cache_path, "constraint_rag_diagnostics.jsonl")

    class constraint_rag_config:
        device = "cuda"
        model_dtype = "cast_bf16"
        max_new_tokens = 32
        retrieval_k = 5

    def run_constraint_rag_experiment(name, config, cache_path=diagnostic_cache_path):
        model_rag = ConstraintRAG(config=config, cache_path=cache_path)
        model_rag.set_model(llm_model)
        model_rag.set_tokenizer(llm_tokenizer)
        model_rag.set_retriever(searcher_bm25)
        print(f"Evaluating {name} on {len(eval_dataset)} examples")
        metrics = eval_for_rag(model_rag, eval_loader, constraint_rag_config, llm_tokenizer)
        prediction = metrics.pop("prediction")
        answers = metrics.pop("answers")
        uid = metrics.pop("uid")
        questions = metrics.pop("question")
        elapsed_seconds = metrics.pop("elapsed_seconds", None)
        del model_rag
        torch.cuda.empty_cache()
        return prediction, answers, uid, questions, elapsed_seconds

    def variant_predictions_from_full(full_prediction):
        return {
            "full": full_prediction,
            "answer_only": [strip_evidence_candidates(pred) for pred in full_prediction],
            "evidence_only": [get_evidence_candidates_text(pred) for pred in full_prediction],
            "first_candidate": [extract_first_answer_candidate(pred) for pred in full_prediction],
        }

    def metadata(result_prefix, description, config, elapsed_seconds=None, extra=None):
        data = {
            "num_samples": len(eval_dataset),
            "retrieval_top_k": constraint_rag_config.retrieval_k,
            "final_top_k": config.final_k,
            "max_new_tokens": constraint_rag_config.max_new_tokens,
            "elapsed_seconds": elapsed_seconds,
            "timing_scope": "eval_for_rag wall time including generation and metric computation",
            "diagnostic_cache_path": diagnostic_cache_path,
            "answer_mode": config.answer_mode,
            "append_evidence_candidates": config.append_evidence_candidates,
            "evidence_candidate_count": config.evidence_candidate_count,
            "evidence_sentence_count": config.evidence_sentence_count,
            "variant": result_prefix,
            "variant_description": description,
        }
        if extra:
            data.update(extra)
        return data

    def write_constraint_variant(result_prefix, prediction, raw_prediction, answers, uid, questions, metrics):
        with open(os.path.join(RESULTS_PATH, f"{result_prefix}_score.json"), "w") as f:
            json.dump(metrics, f, indent=4)
        write_constraint_output(
            os.path.join(RESULTS_PATH, f"{result_prefix}_output.txt"),
            uid,
            questions,
            prediction,
            raw_prediction,
            answers,
        )

    all_constraint_metrics = {}

    main_config = ConstraintRAGConfig(
        final_k=3,
        max_constraints=3,
        decomposition_max_new_tokens=96,
        diagnostic_max_new_tokens=96,
        generation_batch_size=8,
        evidence_candidate_count=10,
        evidence_sentence_count=2,
        evidence_sentence_max_chars=160,
        append_evidence_candidates=True,
        answer_mode="candidate_list",
    )
    raw_prediction, answers, uid, questions, elapsed_seconds = run_constraint_rag_experiment("ConstraintRAG candidate-generation suite", main_config)
    full_prediction = [extract_answer_from_json(pred) for pred in raw_prediction]
    predictions = variant_predictions_from_full(full_prediction)
    main_specs = {
        "constraint_rag_first_candidate": (predictions["first_candidate"], "ConstraintRAG-FirstCandidate: first generated answer candidate only"),
        "constraint_rag_answer_only": (predictions["answer_only"], "ConstraintRAG-Candidates: generated candidate list before evidence suffix"),
        "constraint_rag_evidence_only": (predictions["evidence_only"], "ConstraintRAG-Evidence: only the appended evidence candidates"),
        "constraint_rag": (predictions["full"], "ConstraintRAG-Recall: full generated candidate list plus evidence candidates"),
    }
    main_metrics = {}
    print("====== ConstraintRAG output variants (FirstCandidate first) ====")
    for result_prefix, (prediction, description) in main_specs.items():
        scores = compute_rag_metrics(prediction, answers, rouge, metadata(result_prefix, description, main_config, elapsed_seconds))
        main_metrics[result_prefix] = scores
        all_constraint_metrics[result_prefix] = scores
        write_constraint_variant(result_prefix, prediction, raw_prediction, answers, uid, questions, scores)
        print(f"   {result_prefix}: accuracy={scores['accuracy']:.4f}, rouge1={scores['rouge1']:.4f}, rougeL={scores['rougeL']:.4f}")
    print("================================================")
    with open(os.path.join(RESULTS_PATH, "constraint_rag_variant_scores.json"), "w") as f:
        json.dump(main_metrics, f, indent=4)

    ablation_experiments = [
        {
            "prefix": "constraint_rag_final_k2",
            "config": ConstraintRAGConfig(
                final_k=2,
                max_constraints=3,
                decomposition_max_new_tokens=96,
                diagnostic_max_new_tokens=96,
                generation_batch_size=8,
                evidence_candidate_count=10,
                evidence_sentence_count=2,
                evidence_sentence_max_chars=160,
                append_evidence_candidates=True,
                answer_mode="candidate_list",
            ),
            "description": "ConstraintRAG with final_top_k=2 and evidence candidates enabled",
            "variants": ["full", "answer_only", "evidence_only", "first_candidate"],
        },
        {
            "prefix": "constraint_rag_no_evidence",
            "config": ConstraintRAGConfig(
                final_k=3,
                max_constraints=3,
                decomposition_max_new_tokens=96,
                diagnostic_max_new_tokens=96,
                generation_batch_size=8,
                evidence_candidate_count=10,
                evidence_sentence_count=0,
                evidence_sentence_max_chars=160,
                append_evidence_candidates=False,
                answer_mode="candidate_list",
            ),
            "description": "ConstraintRAG-NoEvidence with final_top_k=3 and evidence candidates disabled",
            "variants": ["full", "first_candidate"],
        },
    ]
    print("====== ConstraintRAG ablation checks ====")
    for experiment in ablation_experiments:
        config = experiment["config"]
        raw_prediction, answers, uid, questions, elapsed_seconds = run_constraint_rag_experiment(experiment["prefix"], config)
        full_prediction = [extract_answer_from_json(pred) for pred in raw_prediction]
        predictions = variant_predictions_from_full(full_prediction)
        for variant in experiment["variants"]:
            result_prefix = experiment["prefix"] if variant == "full" else f"{experiment['prefix']}_{variant}"
            description = experiment["description"] if variant == "full" else f"{experiment['description']} ({variant})"
            prediction = predictions[variant]
            scores = compute_rag_metrics(prediction, answers, rouge, metadata(result_prefix, description, config, elapsed_seconds))
            all_constraint_metrics[result_prefix] = scores
            write_constraint_variant(result_prefix, prediction, raw_prediction, answers, uid, questions, scores)
            print(f"   {result_prefix}: accuracy={scores['accuracy']:.4f}, rouge1={scores['rouge1']:.4f}, rougeL={scores['rougeL']:.4f}")
    print("=========================================")
    direct_config = ConstraintRAGConfig(
        final_k=3,
        max_constraints=3,
        decomposition_max_new_tokens=96,
        diagnostic_max_new_tokens=96,
        generation_batch_size=8,
        evidence_candidate_count=10,
        evidence_sentence_count=2,
        evidence_sentence_max_chars=160,
        append_evidence_candidates=False,
        answer_mode="short_answer",
    )
    prediction, answers, uid, questions, elapsed_seconds = run_constraint_rag_experiment("ConstraintRAG-Direct", direct_config)
    scores = compute_rag_metrics(
        prediction,
        answers,
        rouge,
        metadata(
            "constraint_rag_direct",
            "ConstraintRAG-Direct: constraint-guided passage selection with direct short-answer generation",
            direct_config,
            elapsed_seconds,
        ),
    )
    all_constraint_metrics["constraint_rag_direct"] = scores
    with open(os.path.join(RESULTS_PATH, "constraint_rag_direct_score.json"), "w") as f:
        json.dump(scores, f, indent=4)
    write_rag_output(os.path.join(RESULTS_PATH, "constraint_rag_direct_output.txt"), uid, questions, prediction, answers)
    print("====== constraint_rag_direct evaluation ====")
    for key, value in scores.items():
        print(f"   {key}: {value}")
    print("============================================")

    direct_k2_config = ConstraintRAGConfig(
        final_k=2,
        max_constraints=3,
        decomposition_max_new_tokens=96,
        diagnostic_max_new_tokens=96,
        generation_batch_size=8,
        evidence_candidate_count=10,
        evidence_sentence_count=2,
        evidence_sentence_max_chars=160,
        append_evidence_candidates=False,
        answer_mode="short_answer",
    )
    prediction, answers, uid, questions, elapsed_seconds = run_constraint_rag_experiment("ConstraintRAG-Direct k2", direct_k2_config)
    scores = compute_rag_metrics(
        prediction,
        answers,
        rouge,
        metadata(
            "constraint_rag_direct_final_k2",
            "ConstraintRAG-Direct k2: constraint-guided passage selection with direct short-answer generation, final_top_k=2",
            direct_k2_config,
            elapsed_seconds,
        ),
    )
    all_constraint_metrics["constraint_rag_direct_final_k2"] = scores
    with open(os.path.join(RESULTS_PATH, "constraint_rag_direct_final_k2_score.json"), "w") as f:
        json.dump(scores, f, indent=4)
    write_rag_output(os.path.join(RESULTS_PATH, "constraint_rag_direct_final_k2_output.txt"), uid, questions, prediction, answers)
    print("====== constraint_rag_direct_final_k2 evaluation ====")
    for key, value in scores.items():
        print(f"   {key}: {value}")
    print("=====================================================")

    with open(os.path.join(RESULTS_PATH, "constraint_rag_parameter_ablation_scores.json"), "w") as f:
        json.dump(all_constraint_metrics, f, indent=4)

    del eval_dataset, eval_loader
    torch.cuda.empty_cache()


In [ ]:
if DO_PROPOSAL_BASELINES:
    from huggingface_hub import get_token
    if get_token() is None:
        raise RuntimeError("Hugging Face token not found. Run `hf auth login` before proposal baseline evaluation.")
    from dataset.rag import donwload_dataset_nq_open_dpr, RAGDataset, RAGCollator
    from pyserini.search.lucene import LuceneSearcher
    from proposal_baselines import (
        BM25FilterConfig,
        BM25FilteredRAG,
        RelevanceFilterConfig,
        RelevanceOnlyRAG,
    )

    rag_cache_path = os.path.join(LOCAL_CACHE_PATH, "rag")
    os.makedirs(rag_cache_path, exist_ok=True)
    donwload_dataset_nq_open_dpr(rag_cache_path)

    PREBUILT_INDEX_NAME_BM25 = "wikipedia-dpr-100w"
    LOCAL_INDEX_NAME_BM25 = os.path.join(rag_cache_path, "lucene-index.wikipedia-dpr-100w.20210120.d1b9e6")

    if "searcher_bm25" not in globals():
        try:
            searcher_bm25 = LuceneSearcher(index_dir=LOCAL_INDEX_NAME_BM25)
        except:
            import shutil
            searcher_bm25 = LuceneSearcher.from_prebuilt_index(prebuilt_index_name=PREBUILT_INDEX_NAME_BM25)
            index_dir = searcher_bm25.index_dir
            shutil.move(index_dir, LOCAL_INDEX_NAME_BM25)
            searcher_bm25 = LuceneSearcher(index_dir=LOCAL_INDEX_NAME_BM25)

    if "llm_model" not in globals() or "llm_tokenizer" not in globals():
        model_name_or_path = "meta-llama/Llama-3.2-1B-Instruct"
        llm_model = transformers.AutoModelForCausalLM.from_pretrained(model_name_or_path,
                                                                      torch_dtype=torch.bfloat16,
                                                                      device_map={"": 0},
                                                                      low_cpu_mem_usage=True,)
        llm_tokenizer = transformers.AutoTokenizer.from_pretrained(model_name_or_path)
        llm_tokenizer.pad_token = llm_tokenizer.eos_token
        llm_tokenizer.padding_side = "left"

    eval_dataset = RAGDataset(
        tokenizer=llm_tokenizer,
        is_train=False,
        dataset_path=os.path.join(rag_cache_path, "data", "nq_open_dpr", "nq_dev"),
        num_samples=None,
    )
    eval_collator = RAGCollator(
        tokenizer=llm_tokenizer,
        is_train=False,
    )
    eval_loader = torch.utils.data.DataLoader(
        eval_dataset,
        batch_size=4,
        shuffle=False,
        collate_fn=eval_collator,
    )

    class proposal_baseline_config:
        device = "cuda"
        model_dtype = "cast_bf16"
        max_new_tokens = 32
        retrieval_k = 5

    baseline_experiments = [
        {
            "prefix": "bm25_filter_rag",
            "model": BM25FilteredRAG(BM25FilterConfig(final_k=3)),
            "description": "BM25-only top-3 passage filtering baseline with max_new_tokens=32",
            "final_top_k": 3,
        },
        {
            "prefix": "relevance_rag",
            "model": RelevanceOnlyRAG(
                RelevanceFilterConfig(final_k=3, generation_batch_size=8),
                cache_path=os.path.join(rag_cache_path, "relevance_rag_cache.jsonl"),
            ),
            "description": "LLM scalar relevance top-3 filtering baseline without constraint diagnostics, max_new_tokens=32",
            "final_top_k": 3,
        },
    ]

    for experiment in baseline_experiments:
        result_prefix = experiment["prefix"]
        model_rag = experiment["model"]
        model_rag.set_model(llm_model)
        model_rag.set_tokenizer(llm_tokenizer)
        model_rag.set_retriever(searcher_bm25)

        print(f"Evaluating {result_prefix} on {len(eval_dataset)} examples")
        metrics = eval_for_rag(model_rag, eval_loader, proposal_baseline_config, llm_tokenizer)
        prediction = metrics.pop("prediction")
        answers = metrics.pop("answers")
        uid = metrics.pop("uid")
        questions = metrics.pop("question")
        metrics.update({
            "num_samples": len(eval_dataset),
            "retrieval_top_k": proposal_baseline_config.retrieval_k,
            "final_top_k": experiment["final_top_k"],
            "max_new_tokens": proposal_baseline_config.max_new_tokens,
            "variant": result_prefix,
            "variant_description": experiment["description"],
        })

        with open(os.path.join(RESULTS_PATH, f"{result_prefix}_score.json"), "w") as f:
            json.dump(metrics, f, indent=4)
        with open(os.path.join(RESULTS_PATH, f"{result_prefix}_output.txt"), "w") as f:
            for u, q, p, a in zip(uid, questions, prediction, answers):
                answer_text = " | ".join(a) if isinstance(a, list) else str(a)
                q = str(q).replace("\r", " ").replace("\n", " ").strip()
                p = str(p).replace("\r", " ").replace("\n", " ").strip()
                answer_text = str(answer_text).replace("\r", " ").replace("\n", " ").strip()
                f.write(f"{u}\t{q}\t{answer_text}\t{p}\n")

        print(f"====== {result_prefix} evaluation ====")
        for key, value in metrics.items():
            print(f"   {key}: {value}")
        print("======================================")

        del model_rag
        torch.cuda.empty_cache()

    del eval_dataset, eval_loader
    torch.cuda.empty_cache()


In [ ]:
if DO_HARD_NEGATIVE_RAG:
    from huggingface_hub import get_token
    if get_token() is None:
        raise RuntimeError("Hugging Face token not found. Run `hf auth login` before hard-negative RAG evaluation.")
    from dataset.rag import donwload_dataset_nq_open_dpr, RAGDataset, RAGCollator
    from pyserini.search.lucene import LuceneSearcher
    from proposal_baselines import BM25FilterConfig, BM25FilteredRAG, RelevanceFilterConfig, RelevanceOnlyRAG
    import importlib
    import constraint_rag
    constraint_rag = importlib.reload(constraint_rag)
    ConstraintRAGConfig = constraint_rag.ConstraintRAGConfig
    ConstraintRAG = constraint_rag.ConstraintRAG
    from extra_experiments import (
        HardNegativeSearcher,
        compute_rag_metrics,
        constraint_variant_predictions,
        write_constraint_output,
        write_rag_output,
    )

    rag_cache_path = os.path.join(LOCAL_CACHE_PATH, "rag")
    os.makedirs(rag_cache_path, exist_ok=True)
    donwload_dataset_nq_open_dpr(rag_cache_path)

    PREBUILT_INDEX_NAME_BM25 = "wikipedia-dpr-100w"
    LOCAL_INDEX_NAME_BM25 = os.path.join(rag_cache_path, "lucene-index.wikipedia-dpr-100w.20210120.d1b9e6")

    if "searcher_bm25" not in globals():
        try:
            searcher_bm25 = LuceneSearcher(index_dir=LOCAL_INDEX_NAME_BM25)
        except:
            import shutil
            searcher_bm25 = LuceneSearcher.from_prebuilt_index(prebuilt_index_name=PREBUILT_INDEX_NAME_BM25)
            index_dir = searcher_bm25.index_dir
            shutil.move(index_dir, LOCAL_INDEX_NAME_BM25)
            searcher_bm25 = LuceneSearcher(index_dir=LOCAL_INDEX_NAME_BM25)

    if "llm_model" not in globals() or "llm_tokenizer" not in globals():
        model_name_or_path = "meta-llama/Llama-3.2-1B-Instruct"
        llm_model = transformers.AutoModelForCausalLM.from_pretrained(
            model_name_or_path,
            torch_dtype=torch.bfloat16,
            device_map={"": 0},
            low_cpu_mem_usage=True,
        )
        llm_tokenizer = transformers.AutoTokenizer.from_pretrained(model_name_or_path)
        llm_tokenizer.pad_token = llm_tokenizer.eos_token
        llm_tokenizer.padding_side = "left"

    hard_negative_searcher = HardNegativeSearcher(
        searcher_bm25,
        inject_count=2,
        pool_k=30,
    )
    eval_dataset = RAGDataset(
        tokenizer=llm_tokenizer,
        is_train=False,
        dataset_path=os.path.join(rag_cache_path, "data", "nq_open_dpr", "nq_dev"),
    )
    eval_collator = RAGCollator(tokenizer=llm_tokenizer, is_train=False)
    eval_loader = torch.utils.data.DataLoader(
        eval_dataset,
        batch_size=4,
        shuffle=False,
        collate_fn=eval_collator,
    )

    class hard_negative_config:
        device = "cuda"
        model_dtype = "cast_bf16"
        max_new_tokens = 32
        retrieval_k = 5

    hard_negative_experiments = [
        {
            "prefix": "bm25_filter_hard_negative_rag",
            "display": "BM25FilteredRAG-HardNegative",
            "model": BM25FilteredRAG(BM25FilterConfig(final_k=3)),
            "description": "BM25FilteredRAG under injected hard-negative top-k retrieval noise, final_k=3, max_new_tokens=32",
            "final_top_k": 3,
        },
        {
            "prefix": "relevance_hard_negative_rag",
            "display": "RelevanceOnlyRAG-HardNegative",
            "model": RelevanceOnlyRAG(
                RelevanceFilterConfig(final_k=3, generation_batch_size=8),
                cache_path=os.path.join(rag_cache_path, "relevance_hard_negative_rag_cache.jsonl"),
            ),
            "description": "RelevanceOnlyRAG under injected hard-negative top-k retrieval noise, final_k=3, max_new_tokens=32",
            "final_top_k": 3,
        },
    ]

    for experiment in hard_negative_experiments:
        model_rag = experiment["model"]
        model_rag.set_model(llm_model)
        model_rag.set_tokenizer(llm_tokenizer)
        model_rag.set_retriever(hard_negative_searcher)
        print(f"Evaluating {experiment['prefix']} on {len(eval_dataset)} examples")
        metrics = eval_for_rag(model_rag, eval_loader, hard_negative_config, llm_tokenizer)
        prediction = metrics.pop("prediction")
        answers = metrics.pop("answers")
        uid = metrics.pop("uid")
        questions = metrics.pop("question")
        elapsed_seconds = metrics.get("elapsed_seconds")
        metrics.update({
            "num_samples": len(eval_dataset),
            "retrieval_top_k": hard_negative_config.retrieval_k,
            "final_top_k": experiment["final_top_k"],
            "max_new_tokens": hard_negative_config.max_new_tokens,
            "elapsed_seconds": elapsed_seconds,
            "timing_scope": "eval_for_rag wall time including generation and metric computation",
            "hard_negative_inject_count": 2,
            "hard_negative_pool_k": 30,
            "variant": experiment["prefix"],
            "variant_description": experiment["description"],
        })
        with open(os.path.join(RESULTS_PATH, f"{experiment['prefix']}_score.json"), "w") as f:
            json.dump(metrics, f, indent=4)
        write_rag_output(
            os.path.join(RESULTS_PATH, f"{experiment['prefix']}_output.txt"),
            uid,
            questions,
            prediction,
            answers,
        )
        print(f"====== {experiment['prefix']} evaluation ====")
        for key, value in metrics.items():
            print(f"   {key}: {value}")
        print("============================================")
        del model_rag
        torch.cuda.empty_cache()

    constraint_config = ConstraintRAGConfig(
        final_k=3,
        max_constraints=3,
        decomposition_max_new_tokens=96,
        diagnostic_max_new_tokens=96,
        generation_batch_size=8,
        evidence_candidate_count=10,
        evidence_sentence_count=2,
        evidence_sentence_max_chars=160,
        append_evidence_candidates=True,
    )
    diagnostic_cache_path = os.path.join(rag_cache_path, "constraint_rag_hard_negative_diagnostics.jsonl")
    model_rag = ConstraintRAG(config=constraint_config, cache_path=diagnostic_cache_path)
    model_rag.set_model(llm_model)
    model_rag.set_tokenizer(llm_tokenizer)
    model_rag.set_retriever(hard_negative_searcher)

    class constraint_hard_negative_config:
        device = "cuda"
        model_dtype = "cast_bf16"
        max_new_tokens = 32
        retrieval_k = 5

    print(f"Evaluating constraint_rag_hard_negative on {len(eval_dataset)} examples")
    metrics = eval_for_rag(model_rag, eval_loader, constraint_hard_negative_config, llm_tokenizer)
    raw_prediction = metrics.pop("prediction")
    answers = metrics.pop("answers")
    uid = metrics.pop("uid")
    questions = metrics.pop("question")
    elapsed_seconds = metrics.pop("elapsed_seconds", None)
    variants = constraint_variant_predictions(raw_prediction)
    variant_specs = {
        "constraint_rag_hard_negative_first_candidate": (
            variants["first_candidate"],
            "ConstraintRAG-HardNegative-FirstCandidate: first generated answer candidate only under injected hard-negative retrieval noise",
        ),
        "constraint_rag_hard_negative_answer_only": (
            variants["answer_only"],
            "ConstraintRAG hard-negative generated candidate list before evidence suffix",
        ),
        "constraint_rag_hard_negative_evidence_only": (
            variants["evidence_only"],
            "ConstraintRAG hard-negative evidence candidates only",
        ),
        "constraint_rag_hard_negative": (
            variants["full"],
            "ConstraintRAG hard-negative full generated candidate list plus evidence candidates",
        ),
    }
    hard_negative_metrics = {}
    print("====== constraint_rag_hard_negative variant checks ====")
    for result_prefix, (prediction, description) in variant_specs.items():
        variant_metrics = compute_rag_metrics(
            prediction,
            answers,
            rouge,
            {
                "num_samples": len(eval_dataset),
                "retrieval_top_k": constraint_hard_negative_config.retrieval_k,
                "final_top_k": constraint_config.final_k,
                "max_new_tokens": constraint_hard_negative_config.max_new_tokens,
                "elapsed_seconds": elapsed_seconds,
                "timing_scope": "eval_for_rag wall time including generation and metric computation",
                "hard_negative_inject_count": 2,
                "hard_negative_pool_k": 30,
                "diagnostic_cache_path": diagnostic_cache_path,
                "evidence_candidate_count": constraint_config.evidence_candidate_count,
                "evidence_sentence_count": constraint_config.evidence_sentence_count,
                "variant": result_prefix,
                "variant_description": description,
            },
        )
        hard_negative_metrics[result_prefix] = variant_metrics
        with open(os.path.join(RESULTS_PATH, f"{result_prefix}_score.json"), "w") as f:
            json.dump(variant_metrics, f, indent=4)
        write_constraint_output(
            os.path.join(RESULTS_PATH, f"{result_prefix}_output.txt"),
            uid,
            questions,
            prediction,
            raw_prediction,
            answers,
        )
        print(
            f"   {result_prefix}: accuracy={variant_metrics['accuracy']:.4f}, "
            f"rouge1={variant_metrics['rouge1']:.4f}, rougeL={variant_metrics['rougeL']:.4f}"
        )


    direct_config = ConstraintRAGConfig(
        final_k=3,
        max_constraints=3,
        decomposition_max_new_tokens=96,
        diagnostic_max_new_tokens=96,
        generation_batch_size=8,
        evidence_candidate_count=10,
        evidence_sentence_count=2,
        evidence_sentence_max_chars=160,
        append_evidence_candidates=False,
        answer_mode="short_answer",
    )
    direct_model_rag = ConstraintRAG(config=direct_config, cache_path=diagnostic_cache_path)
    direct_model_rag.set_model(llm_model)
    direct_model_rag.set_tokenizer(llm_tokenizer)
    direct_model_rag.set_retriever(hard_negative_searcher)

    print(f"Evaluating constraint_rag_hard_negative_direct on {len(eval_dataset)} examples")
    direct_metrics = eval_for_rag(direct_model_rag, eval_loader, constraint_hard_negative_config, llm_tokenizer)
    direct_prediction = direct_metrics.pop("prediction")
    direct_answers = direct_metrics.pop("answers")
    direct_uid = direct_metrics.pop("uid")
    direct_questions = direct_metrics.pop("question")
    direct_elapsed_seconds = direct_metrics.pop("elapsed_seconds", None)
    direct_scores = compute_rag_metrics(
        direct_prediction,
        direct_answers,
        rouge,
        {
            "num_samples": len(eval_dataset),
            "retrieval_top_k": constraint_hard_negative_config.retrieval_k,
            "final_top_k": direct_config.final_k,
            "max_new_tokens": constraint_hard_negative_config.max_new_tokens,
            "elapsed_seconds": direct_elapsed_seconds,
            "timing_scope": "eval_for_rag wall time including generation and metric computation",
            "hard_negative_inject_count": 2,
            "hard_negative_pool_k": 30,
            "diagnostic_cache_path": diagnostic_cache_path,
            "answer_mode": direct_config.answer_mode,
            "variant": "constraint_rag_hard_negative_direct",
            "variant_description": "ConstraintRAG-HardNegative-Direct: direct short-answer generation under injected hard-negative retrieval noise",
        },
    )
    hard_negative_metrics["constraint_rag_hard_negative_direct"] = direct_scores
    with open(os.path.join(RESULTS_PATH, "constraint_rag_hard_negative_direct_score.json"), "w") as f:
        json.dump(direct_scores, f, indent=4)
    write_rag_output(
        os.path.join(RESULTS_PATH, "constraint_rag_hard_negative_direct_output.txt"),
        direct_uid,
        direct_questions,
        direct_prediction,
        direct_answers,
    )
    print(
        f"   constraint_rag_hard_negative_direct: accuracy={direct_scores['accuracy']:.4f}, "
        f"rouge1={direct_scores['rouge1']:.4f}, rougeL={direct_scores['rougeL']:.4f}"
    )
    del direct_model_rag
    torch.cuda.empty_cache()
    print("=======================================================")
    with open(os.path.join(RESULTS_PATH, "constraint_rag_hard_negative_variant_scores.json"), "w") as f:
        json.dump(hard_negative_metrics, f, indent=4)

    del eval_dataset, eval_loader, model_rag
    torch.cuda.empty_cache()


In [ ]:
if DO_ZEROSHOT_RAG or DO_FEWSHOT_RAG or DO_CONSTRAINT_RAG or DO_PROPOSAL_BASELINES or DO_HARD_NEGATIVE_RAG or DO_SCRATCH_DOWNSTREAM_BASELINES:
    os.makedirs(RESULTS_PATH, exist_ok=True)
    ablation_files = [
        ("Finetuned small RAG baseline", "rag_score.json"),
        ("Llama zero-shot RAG", "llm_rag_score.json"),
        ("Llama few-shot RAG", "llm_fewshot_rag_score.json"),
        ("BM25FilteredRAG", "bm25_filter_rag_score.json"),
        ("RelevanceOnlyRAG", "relevance_rag_score.json"),
        ("ConstraintRAG-FirstCandidate", "constraint_rag_first_candidate_score.json"),
        ("ConstraintRAG-Direct", "constraint_rag_direct_score.json"),
        ("ConstraintRAG-Direct k2", "constraint_rag_direct_final_k2_score.json"),
        ("ConstraintRAG-NoEvidence", "constraint_rag_no_evidence_first_candidate_score.json"),
        ("ConstraintRAG-FirstCandidate k2", "constraint_rag_final_k2_first_candidate_score.json"),
        ("ConstraintRAG-Candidates", "constraint_rag_answer_only_score.json"),
        ("ConstraintRAG-Candidates k2", "constraint_rag_final_k2_answer_only_score.json"),
        ("ConstraintRAG-Evidence", "constraint_rag_evidence_only_score.json"),
        ("ConstraintRAG-Evidence k2", "constraint_rag_final_k2_evidence_only_score.json"),
        ("ConstraintRAG-Recall", "constraint_rag_score.json"),
        ("ConstraintRAG-Recall k2", "constraint_rag_final_k2_score.json"),
        ("ConstraintRAG-Recall no evidence", "constraint_rag_no_evidence_score.json"),
        ("BM25FilteredRAG-HardNegative", "bm25_filter_hard_negative_rag_score.json"),
        ("RelevanceOnlyRAG-HardNegative", "relevance_hard_negative_rag_score.json"),
        ("ConstraintRAG-HardNegative-FirstCandidate", "constraint_rag_hard_negative_first_candidate_score.json"),
        ("ConstraintRAG-HardNegative-Direct", "constraint_rag_hard_negative_direct_score.json"),
        ("ConstraintRAG-HardNegative-Recall", "constraint_rag_hard_negative_score.json"),
        ("Summary scratch no pretraining", "summary_scratch_score.json"),
        ("Classification scratch no pretraining", "classification_scratch_score.json"),
        ("RAG scratch no pretraining", "rag_scratch_score.json"),
    ]

    ablation_rows = []
    for method, filename in ablation_files:
        path = os.path.join(RESULTS_PATH, filename)
        if not os.path.exists(path):
            print(f"Missing variant score file: {filename}")
            continue
        with open(path) as f:
            score = json.load(f)
        num_samples = score.get("num_samples")
        output_path = os.path.join(RESULTS_PATH, filename.replace("_score.json", "_output.txt"))
        if num_samples is None and os.path.exists(output_path):
            with open(output_path) as output_f:
                num_samples = sum(1 for _ in output_f)
        variant_description = score.get("variant_description", "")
        legacy_prefix = "Pro" + "posed method"
        if variant_description.startswith(legacy_prefix):
            replacement_name = "ConstraintRAG-NoEvidence" if "no_evidence" in filename else "ConstraintRAG"
            variant_description = variant_description.replace(legacy_prefix, replacement_name, 1)
        row = {
            "method": method,
            "score_file": filename,
            "num_samples": num_samples,
            "accuracy": score.get("accuracy"),
            "rouge1": score.get("rouge1"),
            "rouge2": score.get("rouge2"),
            "rougeL": score.get("rougeL"),
            "rougeLsum": score.get("rougeLsum"),
            "retrieval_top_k": score.get("retrieval_top_k"),
            "final_top_k": score.get("final_top_k"),
            "max_new_tokens": score.get("max_new_tokens"),
            "elapsed_seconds": score.get("elapsed_seconds"),
            "timing_scope": score.get("timing_scope"),
            "answer_mode": score.get("answer_mode"),
            "hard_negative_inject_count": score.get("hard_negative_inject_count"),
            "hard_negative_pool_k": score.get("hard_negative_pool_k"),
            "variant_description": variant_description,
        }
        ablation_rows.append(row)

    with open(os.path.join(RESULTS_PATH, "constraint_rag_ablation_summary.json"), "w") as f:
        json.dump(ablation_rows, f, indent=4)

    columns = [
        "method", "num_samples", "accuracy", "rouge1", "rouge2", "rougeL",
        "rougeLsum", "retrieval_top_k", "final_top_k", "max_new_tokens",
        "elapsed_seconds", "timing_scope", "answer_mode",
        "hard_negative_inject_count", "hard_negative_pool_k",
        "score_file", "variant_description"
    ]
    def format_tsv_value(value):
        return "" if value is None else str(value)

    with open(os.path.join(RESULTS_PATH, "constraint_rag_ablation_summary.tsv"), "w") as f:
        f.write("\t".join(columns) + "\n")
        for row in ablation_rows:
            f.write("\t".join(format_tsv_value(row.get(column, "")) for column in columns) + "\n")

    def format_metric(value):
        if value is None:
            return "-"
        try:
            return f"{float(value):.4f}"
        except (TypeError, ValueError):
            return str(value)

    method_width = max(34, max(len(row["method"]) for row in ablation_rows))

    print("====== ConstraintRAG result summary ====")
    for row in ablation_rows:
        print(
            f"{row['method']:<{method_width}s} "
            f"n={str(row['num_samples']):>5s} "
            f"acc={format_metric(row['accuracy']):>8s} "
            f"rouge1={format_metric(row['rouge1']):>8s} "
            f"rougeL={format_metric(row['rougeL']):>8s}"
        )
    print("================================================")


In [ ]:
if DO_SUBMISSION:
    os.makedirs("submission", exist_ok=True)
    os.makedirs("submission/code", exist_ok=True)
    submission_code_list=[
        "main.ipynb",
        "model.py",
        "model_rag.py",
        "constraint_rag.py",
        "proposal_baselines.py",
        "extra_experiments.py",
        "dataset/summary.py",
        "dataset/rag.py",
        "dataset/classification.py",
        "dataset/pretrain.py",
        "utils/etc.py",
        "utils/logger.py",
        "utils/metrics.py",
        "README.md",
    ]
    submission_directory_list=[
        "logs",
        RESULTS_PATH,
        os.path.join(OUTPUT_PATH, "pretraining", "best_model"),
        os.path.join(OUTPUT_PATH, "summary", "best_model"),
        os.path.join(OUTPUT_PATH, "classification", "best_model"),
        os.path.join(OUTPUT_PATH, "rag", "best_model"),
    ]

    for file in submission_code_list:
        dst = os.path.join("submission", "code", file)
        dst_dir = os.path.dirname(dst)
        if not os.path.exists(dst_dir):
            os.makedirs(dst_dir, exist_ok=True)
        if not os.path.exists(file):
            with open(dst, "w") as f:
                f.write(f"{file} not exist")
        else:
            shutil.copyfile(file, dst)

    for directory in submission_directory_list:
        base_name = os.path.basename(directory)
        if not os.path.exists(directory):
            pass
        else:
            if 'best_model' in directory:
                rel_directory = os.path.relpath(directory, PROJ_PATH)
                dst = os.path.join("submission", rel_directory)
            else:
                dst = os.path.join("submission", base_name)
            shutil.copytree(
                directory,
                dst,
                dirs_exist_ok=True,
            )
    shutil.make_archive("defaultproject_code", 'zip', "submission/code")
    shutil.rmtree("submission/code", ignore_errors=True)
    shutil.make_archive("defaultproject_supplementaries", 'zip', "submission")
    shutil.rmtree("submission", ignore_errors=True)
    # With Colab, move to Google Drive, otherwise there are no changes.
    shutil.move("defaultproject_code.zip", os.path.join(PROJ_PATH, "defaultproject_code.zip"))
    shutil.move("defaultproject_supplementaries.zip", os.path.join(PROJ_PATH, "defaultproject_supplementaries.zip"))